# 主成分分析

In [10]:
import numpy as np
from scipy.stats import zscore
from sklearn.decomposition import PCA
import pandas as pd

# 1. 读取原始数据
# 假设 data10_2.txt 是一个包含原始数据的文本文件，例如CSV或TSV格式
# 这里使用 numpy.loadtxt，如果文件是其他格式，可能需要使用 pandas.read_csv
try:
    a = np.loadtxt("data10_2.txt")
except FileNotFoundError:
    print("错误：data10_2.txt 文件未找到。请确保文件存在于当前目录下。")
    # 为了演示，如果文件不存在，创建一个示例数据
    a = np.array([
        [0.71, 0.49, 0.41, 0.51, 0.46],
        [0.40, 0.49, 0.44, 0.57, 0.50],
        [0.55, 0.56, 0.48, 0.53, 0.49],
        [0.62, 0.93, 0.38, 0.53, 0.47],
        [0.45, 0.42, 0.41, 0.54, 0.47],
        [0.36, 0.37, 0.46, 0.54, 0.48],
        [0.55, 0.68, 0.42, 0.54, 0.46],
        [0.62, 0.90, 0.38, 0.56, 0.46],
        [0.61, 0.99, 0.33, 0.57, 0.43],
        [0.71, 0.93, 0.35, 0.66, 0.44],
        [0.59, 0.69, 0.36, 0.57, 0.48],
        [0.41, 0.47, 0.40, 0.54, 0.48],
        [0.26, 0.29, 0.43, 0.57, 0.48],
        [0.14, 0.16, 0.43, 0.55, 0.47],
        [0.12, 0.13, 0.45, 0.59, 0.54],
        [0.22, 0.25, 0.44, 0.58, 0.52],
        [0.71, 0.49, 0.41, 0.51, 0.46]
    ])
    print("使用示例数据进行演示。")

# 2. 数据标准化 (Z-score)
# scipy.stats.zscore 默认对列进行标准化
b = zscore(a)

# 3. 计算相关系数矩阵
r = np.corrcoef(b, rowvar=False) # rowvar=False 表示列代表变量

# 4. 主成分分析 (PCA)
# Matlab 的 pcacov(R) 返回特征向量、特征值和解释方差比例
# sklearn.decomposition.PCA 拟合后可以获取这些信息
# 注意：sklearn 的 PCA 默认是基于样本的协方差矩阵，但如果输入是相关系数矩阵，
# 并且数据已经标准化，效果是等价的。
# 这里我们直接对标准化后的数据 b 进行 PCA，这等同于对相关系数矩阵进行 PCA
# 因为标准化后的数据的协方差矩阵就是相关系数矩阵。

pca = PCA()
pca.fit(b)

x = pca.components_.T # 特征向量 (Matlab的x)，需要转置使其列为特征向量
y = pca.explained_variance_ # 特征值 (Matlab的y)
z = pca.explained_variance_ratio_ * 100 # 解释方差比例 (Matlab的z)，转换为百分比

print("\n特征向量 (x):\n", x)
print("\n特征值 (y):\n", y)
print("\n解释方差比例 (z):\n", z)

# 5. 修正特征向量的正负号
# Matlab 的 sign(sum(x)) 对应 numpy.sign(numpy.sum(x, axis=0))
f = np.sign(np.sum(x, axis=0))
x = x * f # 逐元素相乘，修正特征向量

print("\n修正后的特征向量 (x):\n", x)

# 6. 选取主成分的个数
num = 3

# 7. 计算各个主成分的得分
# df = b * x(:,1:num) 对应 numpy 的矩阵乘法和切片
df = np.dot(b, x[:, :num])

print("\n主成分得分 (df):\n", df)

# 8. 计算综合得分
# tf = df * z(1:num)/100 对应 numpy 的矩阵乘法和逐元素除法
# 注意：z 已经是百分比，所以这里不需要再除以100，除非Matlab的z是原始方差
# 根据文档描述，z是解释方差比例，所以这里直接使用
# 如果Matlab的z是原始方差，则需要除以总方差
# 假设Matlab的z是解释方差比例的百分比，所以这里直接用 z[:num] / 100
tf = np.dot(df, z[:num] / 100)

print("\n综合得分 (tf):\n", tf)

# 9. 把得分按照从高到低的次序排列
# [stf,ind]=sort(tf,\'descend\') 对应 numpy.argsort
ind = np.argsort(tf)[::-1] # 获取降序排列的索引
stf = tf[ind] # 根据索引获取排序后的得分

# Matlab 的 stf=stf\', ind=ind\' 只是转置，Python中根据需要决定是否转置
# 这里保持为一维数组，如果需要列向量形式，可以 reshape
stf = stf.reshape(-1, 1)

# 修正年份数据和索引，使其与原始数据行数匹配
years = np.arange(1984, 2001) # 从1984到2000，共17年
ind_0based = ind # ind 已经是0-based索引

print("\n排序后的综合得分 (stf):\n", stf)
print("\n排序索引 (ind):\n", ind_0based)

print("\n排名和综合评价结果：")
# 创建一个DataFrame以便更好地展示结果
results_df = pd.DataFrame({'年份': years[ind_0based.flatten()],'综合评价': stf.flatten(),'排名': np.arange(1, len(stf) + 1)})
print(results_df.to_string(index=False))


特征向量 (x):
 [[-0.49054217 -0.29344022 -0.51089699  0.18963273  0.61342066]
 [-0.52535146  0.04898756 -0.4336599  -0.12174796 -0.72022398]
 [ 0.48705734 -0.28119552 -0.37135078  0.68875986 -0.2672315 ]
 [-0.06705441  0.89811703 -0.14765822  0.38631158  0.13360356]
 [ 0.49158221  0.16064847 -0.62547502 -0.57060501  0.125419  ]]

特征值 (y):
 [3.33022395 1.24137137 0.37206687 0.23992282 0.12891499]

解释方差比例 (z):
 [62.6865684  23.36699053  7.00361169  4.51619427  2.42663511]

修正后的特征向量 (x):
 [[ 0.49054217 -0.29344022  0.51089699  0.18963273 -0.61342066]
 [ 0.52535146  0.04898756  0.4336599  -0.12174796  0.72022398]
 [-0.48705734 -0.28119552  0.37135078  0.68875986  0.2672315 ]
 [ 0.06705441  0.89811703  0.14765822  0.38631158 -0.13360356]
 [-0.49158221  0.16064847  0.62547502 -0.57060501 -0.125419  ]]

主成分得分 (df):
 [[ 0.72035406 -1.68544501 -0.04274836]
 [-1.08824153  0.39377118  0.63863578]
 [-0.95099003 -1.2199014   1.11884226]
 [ 1.58459804 -0.6650259   0.48051123]
 [-0.21547989 -0.45224038 